# Benchmark — Fase 2: trackers (2×2, consistencia)

## ¿Qué decide este notebook?

**Qué tracker conviene** entre `bytetrack` y `botsort`. Como el costo del tracker es
**casi gratis** frente a la detección, no se elige por velocidad sino por
**consistencia** del seguimiento (sin ground-truth): fragmentación y longitud de
tracklet.

## ¿Por qué 2×2 y no solo sobre el detector ganador de la Fase 1?

Porque la consistencia del tracker **depende de qué cajas le da el detector**. Correr
el **2×2 completo** (ambos detectores × ambos trackers) evita la crítica "¿y si el otro
detector daba mejores tracks?" — son solo 2 corridas extra y el resultado queda
inatacable.

## Métricas (sin ground-truth → consistencia, no exactitud)

- **Robustas:** `fps`, `peak_vram_mb` (eficiencia) y `frag_rate`/`tracklet_len`
  (consistencia del tracking).
- **Débiles (suplementarias):** `smoothness`, `mask_iou`, `com_jitter` — confundidas
  por movimiento real, diluidas por clases estáticas y *gameables*. Se reportan con
  reserva.

## Requisitos

Iguales que la Fase 1 (pod/GPU, `.env`, pesos YOLO + SAM3 + los 5 videos). Usa los
**mismos videos** (seed=36) para que ambas fases sean coherentes.

## Paso 1 — Videos y configuraciones (2×2)

In [1]:
from src.core.batch import run_batch
from src.eval.benchmark import aggregate_config, benchmark_videos, comparison_table

videos = benchmark_videos(n=5, seed=36)  # mismos que la Fase 1
print("videos del benchmark (split=2, seed=36):")
for v in videos:
    print(" ", v)

# 2x2: ambos detectores x ambos trackers. (label, detector, tracker)
CONFIGS = [
    ("sam3_text+bytetrack", "sam3_text", "bytetrack"),
    ("sam3_text+botsort", "sam3_text", "botsort"),
    ("yolo_sam3+bytetrack", "yolo_sam3", "bytetrack"),
    ("yolo_sam3+botsort", "yolo_sam3", "botsort"),
]
print(f"\n{len(CONFIGS)} configuraciones (2 detectores × 2 trackers).")

videos del benchmark (split=2, seed=36):
  data/raw/17Abril/video-597_singular_display.mov
  data/raw/17Abril/video-714_singular_display.mov
  data/raw/17Abril/video-836_singular_display.mov
  data/raw/18abril/Cámaras/IMG_9913.MOV
  data/raw/18abril/video-1054_singular_display.mov

4 configuraciones (2 detectores × 2 trackers).


## Paso 2 — Correr el 2×2 y medir

Por cada config se llama a `run_batch` en `mode="tracking"` con:

- **`sampling="auto"` + `max_frames=MAX_FRAMES`** → prefijo contiguo por *streaming*
  (`iter_frames`): memoria acotada aunque el video sea largo.
- **`include_masks=INCLUDE_MASKS`** → si `True`, añade las métricas de máscara
  (suplementarias, casi gratis); ponlo en `False` para correr más ligero.
- **`run_label=f"trackers/<label>"`** → salidas aisladas por config + skip-done por
  config (reanudable).
- **`progress=True`** → barra de progreso por video (útil para ver el avance del
  tracking en videos largos).

`aggregate_config` produce una fila por config; se persiste al CSV apenas se calcula.

In [2]:
import gc

import pandas as pd
import torch

from src.utils import PROJECT_ROOT

CSV_PATH = PROJECT_ROOT / "outputs" / "benchmark" / "trackers.csv"
MAX_FRAMES = 2500  # tope contiguo (streaming) para acotar videos largos. None = completo.
INCLUDE_MASKS = True  # métricas de máscara (suplementarias). False = más ligero.
RESUME = False  # True para reanudar tras un crash (salta las configs ya guardadas).

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
if not RESUME and CSV_PATH.exists():
    CSV_PATH.unlink()
done = set(pd.read_csv(CSV_PATH)["config"]) if CSV_PATH.exists() else set()
if done:
    print("reanudando; ya guardadas:", done)

for label, det, trk in CONFIGS:
    if label in done:
        print(f"skip {label} (ya en el CSV)")
        continue
    print(f"\n===== {label}  (tracking) =====")
    summary = run_batch(
        mode="tracking",
        videos=videos,
        detector=det,
        tracker=trk,
        sampling="auto",  # prefijo contiguo (streaming)
        max_frames=MAX_FRAMES,
        include_masks=INCLUDE_MASKS,
        render_video=False,
        overwrite=not RESUME,
        run_label=f"trackers/{label}",
        progress=True,
    )
    row = aggregate_config(label, summary)
    comparison_table([row]).to_csv(
        CSV_PATH, mode="a", header=not CSV_PATH.exists(), index=False
    )
    print("fila guardada:", row)

    del summary
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


===== sam3_text+bytetrack  (tracking) =====
== batch: 5 video(s), mode=tracking, render_video=False ==


Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

track video-597_singular_display:   0%|          | 0/641 [00:00<?, ?frame/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

[1/5] data/raw/17Abril/video-597_singular_display.mov -> done


...

[3/5] data/raw/17Abril/video-836_singular_display.mov -> done


track IMG_9913:   0%|          | 0/595 [00:00<?, ?frame/s]

[4/5] data/raw/18abril/Cámaras/IMG_9913.MOV -> done


track video-1054_singular_display:   0%|          | 0/2500 [00:00<?, ?frame/s]

[5/5] data/raw/18abril/video-1054_singular_display.mov -> done
== batch: 5 done, 0 skipped, 0 failed (de 5) ==
fila guardada: {'config': 'yolo_sam3+botsort', 'tracklet_len': 146.47247245763356, 'frag_rate': 0.011435378201310303, 'smoothness': 29.264220218699954, 'mask_iou': 0.9223778329026697, 'com_jitter': 0.002046325543163818, 'fps': 1.9456472620367407, 'peak_vram_mb': 3151.3494528}


## Paso 3 — Tabla comparativa

In [3]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f"tabla de trackers ({len(df)}/{len(CONFIGS)} configs)")
df

tabla de trackers (4/4 configs)


,config,fps,peak_vram_mb,tracklet_len,frag_rate,smoothness,mask_iou,com_jitter
0,sam3_text+bytetrack,2.149065,2157.252915,192.732265,0.034829,261.591224,0.919535,0.002927
1,sam3_text+botsort,1.830581,2157.252915,134.497733,0.060868,246.656169,0.923168,0.002146
2,yolo_sam3+bytetrack,2.248360,3151.165952,186.256143,0.034815,156.695275,0.919036,0.002749
3,yolo_sam3+botsort,1.945647,3151.349453,146.472472,0.011435,29.264220,0.922378,0.002046


## Cómo leer y decidir

| Columna | Qué mide | Mejor cuando | Peso |
|---|---|---|---|
| `fps` | throughput (frames/s) | **mayor** | eficiencia |
| `peak_vram_mb` | VRAM pico | **menor** | eficiencia |
| `tracklet_len` | frames promedio que vive un `obj_id` | **mayor** | **robusta** |
| `frag_rate` | proxy de cambios de ID | **menor** | **robusta** |
| `smoothness` | varianza de la aceleración del centroide | **menor** | débil |
| `mask_iou` | IoU de máscara entre frames consecutivos | **mayor** | débil |
| `com_jitter` | temblor del centro de masa | **menor** | débil |

### Advertencias

- Son métricas **sin GT**: miden **consistencia y eficiencia**, NO exactitud vs.
  humano. Un `frag_rate` bajo no garantiza que el ID sea *correcto*.
- **Decisión del tracker:** mira sobre todo `frag_rate`/`tracklet_len` (la eficiencia
  entre trackers es casi igual). El 2×2 además muestra si el detector elegido en la
  Fase 1 sigue siendo buen sustrato para el tracker.
- La **decisión final es humana**: la tabla informa, no dictamina un ganador
  automático. La exactitud llegará con la evaluación contra ground-truth.